# 03 — Gold: State Energy Explorer serving tables

Builds SQL-endpoint-visible Delta tables. All ratios use null-safe division; no missing or
suppressed value becomes zero.

## Exact table grains

* `gold.gold_state_month_energy` — one row per period, state, and fuel.
* `gold.gold_state_month_electricity` — one row per period and state.
* `gold.gold_state_population` — one row per state/DC.
* `gold.gold_national_month_summary` — one row per period.
* `gold.gold_data_freshness` — one row per source dataset.

`supply_balance_proxy` is explicitly labeled a proxy because state generation divided by retail
sales is not a physical interchange-adjusted power balance.

## Step 1 — Configure Gold processing

Sets the deployment environment and expected 51-jurisdiction coverage used by the Gold
transformations and validation rules.

In [ ]:
environment = "dev"
expected_jurisdictions = 51

## Step 2 — Load Silver sources and select periods

Creates the Gold schema, enables Delta write optimizations, loads the three Silver tables, and
defines null-safe ratio logic. Processing is restricted to the latest eight periods common to
both operational and retail-sales data.

In [ ]:
from pyspark.sql import functions as F, Window

if environment not in {"dev", "test", "prod"}:
    raise ValueError("environment must be dev, test, or prod")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

fuel = spark.table("silver.state_month_fuel")
retail = spark.table("silver.state_month_retail_sales")
population = spark.table("silver.state_population")

def ratio(numerator, denominator):
    return F.when(
        numerator.isNotNull() & denominator.isNotNull() & (denominator > 0),
        numerator / denominator,
    )

periods = sorted(
    set(row["period"] for row in fuel.select("period").distinct().collect())
    .intersection(row["period"] for row in retail.select("period").distinct().collect())
)
selected_periods = periods[-8:]
if len(selected_periods) != 8:
    raise RuntimeError(f"Gold requires 8 common periods; found {selected_periods}")
fuel = fuel.where(F.col("period").isin(selected_periods))
retail = retail.where(F.col("period").isin(selected_periods))

## Fuel-grain table schema and calculations

`consumption_for_eg_btu` is retained with its EIA source-unit metadata. The normalized
`consumption_for_eg_mmbtu` column is published only after the row-level EIA unit fields are
validated as MWh and MMBtu.

## Step 3 — Build the fuel-level analytical table

Validates EIA row-level units before calculating generation mix, fuel intensity, and
population-normalized generation. The resulting `gold_energy` DataFrame preserves the
period-state-fuel grain and labels source units explicitly.

In [ ]:
def normalized_units(df, column_name):
    return {
        row["unit"]
        for row in (
            df.where(F.col(column_name).isNotNull())
            .select(
                F.lower(
                    F.regexp_replace(F.trim(F.col(column_name)), "[^A-Za-z]", "")
                ).alias("unit")
            )
            .distinct()
            .collect()
        )
    }

def assert_units(df, column_name, accepted, dataset_name):
    actual = normalized_units(df, column_name)
    if not actual:
        raise RuntimeError(f"{dataset_name} returned no row-level unit for {column_name}.")
    unexpected = actual.difference(accepted)
    if unexpected:
        raise RuntimeError(
            f"{dataset_name} returned unsupported {column_name} values: {sorted(unexpected)}"
        )

assert_units(
    fuel,
    "generation_units_raw",
    {"megawatthours", "mwh"},
    "EIA electric power operations",
)
assert_units(
    fuel,
    "consumption_for_eg_btu_units_raw",
    {"mmbtu", "millionbtu", "millionbritishthermalunits"},
    "EIA electric power operations",
)
assert_units(
    retail,
    "retail_sales_units_raw",
    {"megawatthours", "mwh"},
    "EIA retail sales",
)

state_totals = fuel.groupBy("period", "state_code").agg(
    F.sum("generation_mwh").alias("state_generation_mwh")
)
gold_energy = (
    fuel.join(state_totals, ["period", "state_code"])
    .join(population.select("state_code", "population"), "state_code")
    .withColumn("consumption_for_eg_mmbtu", F.col("consumption_for_eg_btu"))
    .withColumn("generation_mix_pct", ratio(F.col("generation_mwh"), F.col("state_generation_mwh")) * 100)
    .withColumn(
        "fuel_intensity_mmbtu_per_mwh",
        ratio(F.col("consumption_for_eg_mmbtu"), F.col("generation_mwh")),
    )
    .withColumn(
        "generation_mwh_per_1000_residents",
        ratio(F.col("generation_mwh"), F.col("population")) * 1000,
    )
    .withColumnRenamed("generation_status", "value_status")
    .withColumn("source", F.lit("EIA v2 electric-power-operational-data; Census PEP Vintage 2021"))
    .withColumn("generation_unit", F.lit("MWh"))
    .withColumn("consumption_unit", F.lit("MMBtu"))
    .select(
        "period", "state_code", "state_name", "fuel_code", "fuel_name", "fuel_category",
        "generation_mwh", "consumption_for_eg_btu", "consumption_for_eg_mmbtu",
        "generation_mix_pct", "fuel_intensity_mmbtu_per_mwh", "population",
        "generation_mwh_per_1000_residents", "value_status", "source",
        "generation_unit", "consumption_unit",
    )
)

## Step 4 — Build state-month electricity metrics

Aggregates fuel records to state-month totals, joins retail sales and population, and calculates
carbon-free share, fossil dependency, per-person measures, month-over-month change, state rank,
eight-month ranges, and completeness.

In [ ]:
generation_rollup = fuel.groupBy("period", "state_code", "state_name").agg(
    F.sum("generation_mwh").alias("generation_mwh"),
    F.sum(F.when(F.col("fuel_category") == "carbon_free", F.col("generation_mwh"))).alias("carbon_free_mwh"),
    F.sum(F.when(F.col("fuel_category") == "fossil", F.col("generation_mwh"))).alias("fossil_mwh"),
    F.count("*").alias("fuel_rows"),
    F.sum(F.when(F.col("generation_status") == "reported", 1).otherwise(0)).alias("reported_fuel_rows"),
)

state_month = (
    generation_rollup.join(retail, ["period", "state_code", "state_name"], "full")
    .join(population.select("state_code", "population"), "state_code")
    .withColumn("generation_mwh_per_1000_residents", ratio(F.col("generation_mwh"), F.col("population")) * 1000)
    .withColumn("electricity_consumption_kwh_per_person", ratio(F.col("retail_sales_mwh") * 1000, F.col("population")))
    .withColumn("carbon_free_share_pct", ratio(F.col("carbon_free_mwh"), F.col("generation_mwh")) * 100)
    .withColumn("fossil_dependency_pct", ratio(F.col("fossil_mwh"), F.col("generation_mwh")) * 100)
    .withColumn("supply_balance_proxy", ratio(F.col("generation_mwh"), F.col("retail_sales_mwh")))
)

state_window = Window.partitionBy("state_code").orderBy("period")
period_window = Window.partitionBy("period")
state_month = (
    state_month
    .withColumn("previous_retail_sales_mwh", F.lag("retail_sales_mwh").over(state_window))
    .withColumn(
        "retail_sales_mom_pct",
        ratio(F.col("retail_sales_mwh") - F.col("previous_retail_sales_mwh"), F.col("previous_retail_sales_mwh")) * 100,
    )
    .withColumn("retail_sales_eight_month_high_mwh", F.max("retail_sales_mwh").over(Window.partitionBy("state_code")))
    .withColumn("retail_sales_eight_month_low_mwh", F.min("retail_sales_mwh").over(Window.partitionBy("state_code")))
    .withColumn("retail_sales_state_rank", F.dense_rank().over(period_window.orderBy(F.desc_nulls_last("retail_sales_mwh"))))
    .withColumn("retail_sales_national_median_mwh", F.percentile_approx("retail_sales_mwh", F.lit(0.5)).over(period_window))
    .withColumn(
        "completeness_pct",
        (
            F.col("reported_fuel_rows")
            + F.when(F.col("retail_sales_status") == "reported", 1).otherwise(0)
        ) / (F.col("fuel_rows") + 1) * 100,
    )
    .withColumn("source", F.lit("EIA v2 operational + retail-sales; Census PEP Vintage 2021"))
    .select(
        "period", "state_code", "state_name", "population", "generation_mwh",
        "retail_sales_mwh", "generation_mwh_per_1000_residents",
        "electricity_consumption_kwh_per_person", "carbon_free_share_pct",
        "fossil_dependency_pct", "supply_balance_proxy", "retail_sales_mom_pct",
        "retail_sales_eight_month_high_mwh", "retail_sales_eight_month_low_mwh",
        "retail_sales_state_rank", "retail_sales_national_median_mwh",
        "completeness_pct", "source",
    )
)

## Step 5 — Build national and population outputs

Rolls state-month values into national monthly summaries with coverage metrics and creates the
state population serving table. Both outputs retain source and retrieval metadata needed by the
application and SQL endpoint consumers.

In [ ]:
national = (
    state_month.groupBy("period")
    .agg(
        F.sum("population").alias("population"),
        F.sum("generation_mwh").alias("generation_mwh"),
        F.sum("retail_sales_mwh").alias("retail_sales_mwh"),
        F.sum(F.col("generation_mwh") * F.col("carbon_free_share_pct") / 100).alias("carbon_free_mwh"),
        F.sum(F.col("generation_mwh") * F.col("fossil_dependency_pct") / 100).alias("fossil_mwh"),
        F.countDistinct("state_code").alias("reporting_jurisdictions"),
    )
    .withColumn("expected_jurisdictions", F.lit(expected_jurisdictions))
    .withColumn("electricity_consumption_kwh_per_person", ratio(F.col("retail_sales_mwh") * 1000, F.col("population")))
    .withColumn("carbon_free_share_pct", ratio(F.col("carbon_free_mwh"), F.col("generation_mwh")) * 100)
    .withColumn("fossil_dependency_pct", ratio(F.col("fossil_mwh"), F.col("generation_mwh")) * 100)
    .withColumn("supply_balance_proxy", ratio(F.col("generation_mwh"), F.col("retail_sales_mwh")))
    .withColumn("previous_retail_sales_mwh", F.lag("retail_sales_mwh").over(Window.orderBy("period")))
    .withColumn(
        "retail_sales_mom_pct",
        ratio(F.col("retail_sales_mwh") - F.col("previous_retail_sales_mwh"), F.col("previous_retail_sales_mwh")) * 100,
    )
    .withColumn("completeness_pct", F.col("reporting_jurisdictions") / F.col("expected_jurisdictions") * 100)
    .withColumn("source", F.lit("EIA v2 operational + retail-sales; Census PEP Vintage 2021"))
    .select(
        "period", "population", "generation_mwh", "retail_sales_mwh",
        "electricity_consumption_kwh_per_person", "carbon_free_share_pct",
        "fossil_dependency_pct", "supply_balance_proxy", "retail_sales_mom_pct",
        "reporting_jurisdictions", "expected_jurisdictions", "completeness_pct", "source",
    )
)

gold_population = (
    population.select(
        "state_code", "state_name", "census_state_fips", "population", "estimate_year",
        "vintage", F.col("source_url").alias("source"),
        F.col("ingested_at_utc").alias("retrieved_at_utc"),
    )
)

## Step 6 — Build data-freshness metadata

Calculates the latest available period, retrieval timestamp, jurisdiction coverage, completeness,
and status for each EIA and Census source. These rows power transparent freshness indicators in
the State Energy Explorer.

In [ ]:
retrieval = fuel.agg(F.max("ingested_at_utc").alias("retrieved")).first()["retrieved"]
freshness_rows = []
for dataset_name, frame, source in [
    ("EIA operational", fuel, "EIA v2 electric-power-operational-data"),
    ("EIA retail sales", retail, "EIA v2 retail-sales"),
]:
    latest = frame.agg(F.max("period").alias("latest")).first()["latest"]
    actual = (
        frame.where(F.col("period") == selected_periods[-1])
        .select("state_code").distinct().count()
    )
    completeness = actual / expected_jurisdictions * 100
    freshness_rows.append(
        (
            dataset_name, source, latest, selected_periods[-1], selected_periods[0],
            retrieval, expected_jurisdictions, actual, completeness,
            "complete" if actual == expected_jurisdictions else "incomplete",
            None if actual == expected_jurisdictions else "Latest selected period has incomplete jurisdiction coverage.",
        )
    )
freshness_rows.append(
    (
        "Census population", "Census PEP Vintage 2021", None, selected_periods[-1],
        selected_periods[0], population.agg(F.max("ingested_at_utc")).first()[0],
        expected_jurisdictions, population.select("state_code").distinct().count(),
        population.select("state_code").distinct().count() / expected_jurisdictions * 100,
        "complete" if population.select("state_code").distinct().count() == expected_jurisdictions else "incomplete",
        "Annual population estimate applied to every monthly period.",
    )
)
freshness_schema = (
    "dataset_name string, source string, latest_available_period string, "
    "latest_selected_period string, earliest_selected_period string, retrieved_at_utc timestamp, "
    "expected_jurisdictions int, actual_jurisdictions int, completeness_pct double, "
    "status string, notes string"
)
freshness = spark.createDataFrame(freshness_rows, freshness_schema)

## Step 7 — Validate Gold table grains and ranges

Enforces the documented natural keys, checks nonnegative and percentage ranges, and confirms that
every state-month period contains all 51 jurisdictions before any serving table is published.

In [ ]:
def assert_unique(df, keys, name):
    if df.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count():
        raise RuntimeError(f"{name} has duplicate keys: {keys}")

assert_unique(gold_energy, ["period", "state_code", "fuel_code"], "gold_state_month_energy")
assert_unique(state_month, ["period", "state_code"], "gold_state_month_electricity")
assert_unique(gold_population, ["state_code"], "gold_state_population")
assert_unique(national, ["period"], "gold_national_month_summary")

for name, frame in {
    "gold_state_month_electricity": state_month,
    "gold_national_month_summary": national,
}.items():
    bad = frame.where(
        (F.col("generation_mwh") < 0)
        | (F.col("retail_sales_mwh") < 0)
        | (F.col("completeness_pct") < 0)
        | (F.col("completeness_pct") > 100)
    )
    if bad.limit(1).count():
        raise RuntimeError(f"{name} failed nonnegative/range validation.")
if state_month.groupBy("period").agg(F.countDistinct("state_code").alias("n")).where(F.col("n") != expected_jurisdictions).limit(1).count():
    raise RuntimeError("Gold state-month output does not contain all 51 jurisdictions.")

## Step 8 — Publish Gold serving tables

Writes all five Gold DataFrames as schema-overwriting Delta tables, partitions monthly outputs by
period, applies table optimization, and prints final row counts for reconciliation.

In [ ]:
outputs = {
    "gold.gold_state_month_energy": (gold_energy, ["period"]),
    "gold.gold_state_month_electricity": (state_month, ["period"]),
    "gold.gold_state_population": (gold_population, []),
    "gold.gold_national_month_summary": (national, ["period"]),
    "gold.gold_data_freshness": (freshness, []),
}
for table_name, (frame, partitions) in outputs.items():
    writer = frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partitions:
        writer = writer.partitionBy(*partitions)
    writer.saveAsTable(table_name)

spark.sql("OPTIMIZE gold.gold_state_month_energy ZORDER BY (state_code, fuel_code)")
spark.sql("OPTIMIZE gold.gold_state_month_electricity ZORDER BY (state_code)")
spark.sql("OPTIMIZE gold.gold_national_month_summary")

for table_name in outputs:
    print(table_name, spark.table(table_name).count())